# Case study: the same aggregation in SQL, pandas and Polars

*Session 3, block 3 · Author: course team · Licence: CC-BY-4.0*

**Question.** For each store and year: how many reviews, what average rating, how many words per review,
and what share of reviews mention a *refund*? Keep store-years with at least 500 reviews.

We answer it on all 434,373 training reviews (`train.parquet`, about 70 MB on disk) in three tools and
compare the **code**, the **runtime** and the **memory use**.

Your timings will differ from your neighbours': they depend on the processor, the number of cores, the
disk and the library versions. Compare the *ratios*, not the absolute numbers.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import platform
import time

import duckdb
import pandas as pd
import polars as pl

TRAIN, PRODUCTS = DATA / "train.parquet", DATA / "products.parquet"
print(platform.platform(), "| pandas", pd.__version__, "| polars", pl.__version__, "| duckdb", duckdb.__version__)

## 1. SQL (DuckDB on the Parquet files)

In [ ]:
SQL = f"""
SELECT p.store,
       EXTRACT(YEAR FROM r.date)::int AS year,
       COUNT(*) AS n,
       AVG(r.rating) AS avg_rating,
       AVG(len(regexp_split_to_array(trim(r.text), '\\s+'))) AS avg_words,
       AVG(CASE WHEN r.text ILIKE '%refund%' THEN 1.0 ELSE 0.0 END) AS share_refund
FROM read_parquet('{TRAIN}') AS r
LEFT JOIN read_parquet('{PRODUCTS}') AS p USING (parent_asin)
GROUP BY p.store, year
HAVING COUNT(*) >= 500
ORDER BY year, avg_rating DESC
"""


def run_sql():
    return duckdb.sql(SQL).pl()        # result as a Polars DataFrame


res_sql = run_sql()
res_sql.head()

## 2. pandas (eager)

pandas executes each line immediately (**eager evaluation**): it reads the whole file, then builds every
intermediate table in memory.

In [ ]:
def run_pandas():
    reviews = pd.read_parquet(TRAIN)                                  # all 11 columns
    products = pd.read_parquet(PRODUCTS, columns=["parent_asin", "store"])
    return (
        reviews.merge(products, on="parent_asin", how="left", validate="many_to_one")
        .assign(year=lambda d: d["date"].dt.year,
                n_words=lambda d: d["text"].str.count(r"\S+"),
                refund=lambda d: d["text"].str.contains("refund", case=False))
        .groupby(["store", "year"], as_index=False, dropna=False)     # dropna=False keeps the NULL store
        .agg(n=("rating", "size"), avg_rating=("rating", "mean"),
             avg_words=("n_words", "mean"), share_refund=("refund", "mean"))
        .query("n >= 500")
        .sort_values(["year", "avg_rating"], ascending=[True, False])
    )


res_pd = run_pandas()
res_pd.head()

## 3. Polars: eager and lazy

Polars code is built from **expressions** such as `pl.col("rating").mean()`. With `scan_parquet` instead
of `read_parquet`, nothing is computed yet: Polars builds a **query plan** (a `LazyFrame`), optimises it and
runs it on `.collect()`.

In [ ]:
def build_query(source: pl.LazyFrame, products: pl.LazyFrame) -> pl.LazyFrame:
    return (
        source.join(products.select("parent_asin", "store"), on="parent_asin", how="left", validate="m:1")
        .group_by("store", pl.col("date").dt.year().alias("year"))
        .agg(n=pl.len(),
             avg_rating=pl.col("rating").mean(),
             avg_words=pl.col("text").str.count_matches(r"\S+").mean(),
             share_refund=pl.col("text").str.to_lowercase().str.contains("refund").mean())
        .filter(pl.col("n") >= 500)
        .sort(["year", "avg_rating"], descending=[False, True])
    )


def run_polars_eager():
    # read everything first, then compute (like pandas)
    return build_query(pl.read_parquet(TRAIN).lazy(), pl.read_parquet(PRODUCTS).lazy()).collect()


def run_polars_lazy():
    return build_query(pl.scan_parquet(TRAIN), pl.scan_parquet(PRODUCTS)).collect()


def run_polars_streaming():
    # the streaming engine processes the file in batches; it also works for data larger than memory
    return build_query(pl.scan_parquet(TRAIN), pl.scan_parquet(PRODUCTS)).collect(engine="streaming")


res_pl = run_polars_lazy()
res_pl.head()

### What did the optimiser do?

`explain()` prints the optimised plan. Look for the lines that scan the Parquet files: Polars reads only
the columns it needs (**projection pushdown**: 4 of 11 columns of `train.parquet`). A filter on the input,
for example `verified_purchase == True`, would also be pushed into the scan (**predicate pushdown**).

In [ ]:
print(build_query(pl.scan_parquet(TRAIN), pl.scan_parquet(PRODUCTS)).explain())

## 4. Are the results the same?

Before comparing speed, check that the three tools answer the same question. Small floating-point
differences are expected; the store-year groups and counts must agree exactly.

In [ ]:
from polars.testing import assert_frame_equal

key = ["year", "store"]
a = res_pl.sort(key)
b = pl.from_pandas(res_pd).with_columns(pl.col("year").cast(pl.Int32), pl.col("n").cast(pl.UInt32)).sort(key)
c = res_sql.with_columns(pl.col("n").cast(pl.UInt32)).sort(key)
exact = ["year", "store", "n", "avg_rating", "share_refund"]
assert_frame_equal(a.select(exact), b.select(exact), check_dtypes=False, abs_tol=1e-9)
assert_frame_equal(a.select(exact), c.select(exact), check_dtypes=False, abs_tol=1e-9)
# word counts agree only to about 3 digits: see the next cell
assert_frame_equal(a.select(key + ["avg_words"]), b.select(key + ["avg_words"]), check_dtypes=False, rel_tol=1e-3)
assert_frame_equal(a.select(key + ["avg_words"]), c.select(key + ["avg_words"]), check_dtypes=False, rel_tol=1e-3)
print("groups, counts, ratings and refund shares agree exactly:", a.shape)
pl.DataFrame({"store": a["store"], "year": a["year"], "polars": a["avg_words"],
              "pandas": b["avg_words"], "sql": c["avg_words"]}).filter(pl.col("polars") != pl.col("pandas"))

The word counts differ slightly for some groups. The three regular-expression engines do not agree on
what counts as white space: the character U+00A0 (*no-break space*), which appears in some reviews,
separates words for one engine but not for another. The cell below finds the affected reviews. This is a
typical finding when the same rule is implemented in two tools: decide on one definition (for example,
replace U+00A0 by a normal space during cleaning, Session 4) and document it.

In [ ]:
words = (pl.read_parquet(TRAIN, columns=["review_id", "text"])
         .with_columns(polars_words=pl.col("text").str.count_matches(r"\S+"),
                       has_nbsp=pl.col("text").str.contains("\u00a0", literal=True)))
words = words.with_columns(pandas_words=pl.Series(words["text"].to_pandas().str.count(r"\S+")))
print("reviews with U+00A0:", words["has_nbsp"].sum())
words.filter(pl.col("polars_words") != pl.col("pandas_words")).select(
    "review_id", "polars_words", "pandas_words", "has_nbsp").head()

## 5. Runtime

Each version runs three times; we keep the fastest run (the others include warm-up effects such as the
operating system's file cache).

In [ ]:
def best_of(fn, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


runs = {"SQL (DuckDB)": run_sql, "pandas": run_pandas, "Polars eager": run_polars_eager,
        "Polars lazy": run_polars_lazy, "Polars streaming": run_polars_streaming}
runtime = pd.Series({name: best_of(fn) for name, fn in runs.items()}, name="seconds").round(2)
runtime.to_frame().assign(relative_to_pandas=lambda d: (d["seconds"] / d.loc["pandas", "seconds"]).round(2))

## 6. Memory

**In-memory size of the table.** pandas reports the bytes of each column (`deep=True` includes the
strings); Polars estimates the size of its Arrow buffers.

In [ ]:
reviews_pd = pd.read_parquet(TRAIN)
reviews_pl = pl.read_parquet(TRAIN)
print(f"pandas: {reviews_pd.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"Polars: {reviews_pl.estimated_size() / 1e6:,.0f} MB")
print(f"file on disk (compressed Parquet): {TRAIN.stat().st_size / 1e6:,.0f} MB")
del reviews_pd, reviews_pl

**Peak memory of the whole computation.** Memory that is freed again does not show up in the table size,
so we measure the **peak resident memory** of a fresh Python process for each version. The baseline is a
process that only imports the libraries.

In [ ]:
import subprocess
import sys
import textwrap

BENCH = textwrap.dedent(f"""
    import resource, sys
    import duckdb, pandas as pd, polars as pl
    TRAIN, PRODUCTS = {str(TRAIN)!r}, {str(PRODUCTS)!r}
    version = sys.argv[1]
    if version == "pandas":
        r = pd.read_parquet(TRAIN); p = pd.read_parquet(PRODUCTS, columns=["parent_asin", "store"])
        out = (r.merge(p, on="parent_asin", how="left")
                .assign(year=r["date"].dt.year, n_words=r["text"].str.count(r"\\S+"),
                        refund=r["text"].str.contains("refund", case=False))
                .groupby(["store", "year"], dropna=False)
                .agg(n=("rating", "size"), avg_rating=("rating", "mean"),
                     avg_words=("n_words", "mean"), share_refund=("refund", "mean")))
    elif version.startswith("polars"):
        lazy = version != "polars_eager"
        src = pl.scan_parquet(TRAIN) if lazy else pl.read_parquet(TRAIN).lazy()
        prod = pl.scan_parquet(PRODUCTS) if lazy else pl.read_parquet(PRODUCTS).lazy()
        q = (src.join(prod.select("parent_asin", "store"), on="parent_asin", how="left")
               .group_by("store", pl.col("date").dt.year().alias("year"))
               .agg(n=pl.len(), avg_rating=pl.col("rating").mean(),
                    avg_words=pl.col("text").str.count_matches(r"\\S+").mean(),
                    share_refund=pl.col("text").str.to_lowercase().str.contains("refund").mean()))
        out = q.collect(engine="streaming" if version == "polars_streaming" else "auto")
    elif version == "duckdb":
        out = duckdb.sql(sys.stdin.read()).pl()
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(peak / (1e6 if sys.platform == "darwin" else 1e3))   # bytes on macOS, kB on Linux -> MB
""")


def peak_mb(version: str) -> float:
    result = subprocess.run([sys.executable, "-c", BENCH, version], input=SQL,
                            capture_output=True, text=True, check=True)
    return float(result.stdout.strip().splitlines()[-1])


versions = {"baseline (imports only)": "baseline", "SQL (DuckDB)": "duckdb", "pandas": "pandas",
            "Polars eager": "polars_eager", "Polars lazy": "polars_lazy", "Polars streaming": "polars_streaming"}
peak = pd.Series({name: peak_mb(v) for name, v in versions.items()}, name="peak_MB").round(0)
peak.to_frame().assign(above_baseline=lambda d: d["peak_MB"] - d.loc["baseline (imports only)", "peak_MB"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
runtime.plot.barh(ax=axes[0], color="#2a78d6")
axes[0].set(title="Runtime (best of 3)", xlabel="seconds")
(peak.drop("baseline (imports only)") - peak["baseline (imports only)"]).plot.barh(ax=axes[1], color="#eb6834")
axes[1].set(title="Peak memory above baseline", xlabel="MB")
for ax in axes:
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 7. Compare the code

| Step | SQL | pandas | Polars |
|---|---|---|---|
| read | `FROM read_parquet(...)` | `pd.read_parquet` (eager) | `pl.scan_parquet` (lazy) |
| join | `LEFT JOIN ... USING` | `merge(how="left", validate=...)` | `join(how="left", validate="m:1")` |
| new column | expression in `SELECT` | `assign(...)` | expression inside `agg` |
| group | `GROUP BY` (keeps NULL) | `groupby(dropna=False)` | `group_by` (keeps null) |
| filter groups | `HAVING` | `query` after `agg` | `filter` after `agg` |

**Questions for discussion**

1. Which version was fastest on your machine, and by what factor? Which used the least memory?
2. Remove `dropna=False` from the pandas version. How many rows does the result lose, and why?
3. Add a filter `verified_purchase == True` *before* the join in the Polars query. Where does it appear in
   `explain()`?
4. pandas can also read only the columns it needs: `pd.read_parquet(TRAIN, columns=[...])`. How much does
   that change its memory use?
5. Your project data have 10 million rows and live in a PostgreSQL database shared by the team. Which tool
   would you use for which step, and why?

In [ ]:
# space for questions 2-4